# PDF 문서 기반 RAG — 정답본

원본: `src/solution/Chapter02/00-review/03_langchain_rag.py`  
슬라이드: `docs/raw/Chapter02/day01/3-1-course-one-review-and-advanced-direction.pdf`

## 학습 목표

- 문서 분할·임베딩·검색·답변 연결
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

PDF → 500자 청크 → Chroma → 상위 3개 검색 → 프롬프트 → LLM

**읽을 때 확인할 점:** 검색 문서가 답변의 근거인지 직접 대조한다. while True 입력 루프는 셀을 계속 점유한다.


## 실행 전 확인

동봉 PDF와 임베딩·LLM 접속이 필요하다. 입력 루프는 셀 중단으로 멈춘다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/solution/Chapter02/00-review/03_langchain_rag.py")


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


PDF_PATH = Path(__file__).parent / 'data' / 'meritz_pet_insurance.pdf'
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
        [
            (
                'system',
                '당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.',
            ),
            ('human', 'context:\n{context}\n\nquestion:\n{question}'),
        ]
    )
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)


rag_chain = (
    { 'context': retriever | format_docs, 'question': RunnablePassthrough() }
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG ready')
while True:
    question = input('prompt: ')
    for message in rag_chain.stream(question):
        print(message, end='')
    print('\n')


## CodeMate와 함께 검토

> 질문이 context와 question으로 어떻게 흘러가는지 설명하고 답변의 근거를 검사할 방법을 제안해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. chunk_overlap이 0이면 어떤 영향이 있는가?
2. 검색 문서와 답변이 다르면 어디를 확인하는가?
